# WER Evaluation Pipeline — Multi-Dialect Vietnamese TTS

Evaluate TTS quality across **6 checkpoints** and **3 dialects** by:
1. Generating speech with F5-TTS for each test transcript × dialect
2. Transcribing generated audio with PhoWhisper Medium ASR
3. Computing Word Error Rate (WER) against original transcripts

**Total: 50 samples × 3 dialects × 6 checkpoints = 900 generations**

## 1. Setup & Install

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/mdv-tts/F5-TTS
!pip install -e . -q
!pip install jiwer -q

/content/drive/MyDrive/mdv-tts/F5-TTS
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 14.6 MB/s eta 0:00:00 0:00:01m
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 52.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 67.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 96.2 MB/s eta 0:00:00
   

## 2. Configuration

In [1]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/mdv-tts"
TESTS_CSV = os.path.join(DRIVE_ROOT, "tests", "tests.csv")
VOCAB_FILE = os.path.join(DRIVE_ROOT, "F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt")
CKPT_DIR = os.path.join(DRIVE_ROOT, "F5-TTS/ckpts/viet")
OUTPUT_DIR = "/content/wer_results"
DRIVE_RESULTS = os.path.join(DRIVE_ROOT, "tests", "results")

CHECKPOINTS = [2000, 10000, 20000, 30000, 35000, 40000]
DIALECTS = ["North", "Central", "South"]

# Reference audio & text per dialect (same config as mdv_tts_app.py)
DIALECT_CONFIG = {
    "North": {
        "ref_audio": os.path.join(DRIVE_ROOT, "references/north_ref.wav"),
        "ref_text": (
            "[North] ý tưởng đầu tiên tôi phải nhắc đến là vấn đề về chi phí xây dựng phần mềm biết cốt theo truyền thống."
        ),
    },
    "Central": {
        "ref_audio": os.path.join(DRIVE_ROOT, "references/central_ref.wav"),
        "ref_text": (
            "[Central] các cái ki bốt này được đầu tư từ những năm hai không mười hai không mười ba tức là cách đây cũng khoảng hơn mười năm rồi và quá trình."
        ),
    },
    "South": {
        "ref_audio": os.path.join(DRIVE_ROOT, "references/south_ref.wav"),
        "ref_text": (
            "[South] Chúng tôi tập trung vào cái công tác đào tạo huấn luyện, đặc biệt là huấn luyện về cái quy trình các cái hướng dẫn chẩn đoán điều trị Covid mười chín của Bộ Y tế,"
        ),
    },
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Verify paths
print(f"Tests CSV exists: {os.path.exists(TESTS_CSV)}")
print(f"Vocab file exists: {os.path.exists(VOCAB_FILE)}")
for name, cfg in DIALECT_CONFIG.items():
    print(f"{name} ref audio exists: {os.path.exists(cfg['ref_audio'])}")
for ckpt in CHECKPOINTS:
    p = os.path.join(CKPT_DIR, f"model_{ckpt}.safetensors")
    print(f"Checkpoint {ckpt} exists: {os.path.exists(p)}")

Tests CSV exists: True
Vocab file exists: True
North ref audio exists: True
Central ref audio exists: True
South ref audio exists: True
Checkpoint 2000 exists: False
Checkpoint 10000 exists: False
Checkpoint 20000 exists: True
Checkpoint 30000 exists: True
Checkpoint 35000 exists: True
Checkpoint 40000 exists: True


## 3. Load PhoWhisper ASR Model

In [ ]:
import torch
from transformers import pipeline as hf_pipeline

print("Loading PhoWhisper Medium...")
asr_pipe = hf_pipeline(
    "automatic-speech-recognition",
    model="vinai/PhoWhisper-medium",
    device="cuda:0",
    torch_dtype=torch.float16,
)
print("PhoWhisper loaded!")

## 4. Utility Functions

In [ ]:
import re
import csv
import gc
import time
import tempfile
import numpy as np
import soundfile as sf
from jiwer import wer as compute_wer


def normalize_text(text: str) -> str:
    """Normalize text for WER comparison.
    Removes dialect tags, punctuation, extra whitespace, and lowercases.
    """
    text = re.sub(r"\[(North|Central|South)\]\s*", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def load_test_samples(csv_path: str) -> list[dict]:
    """Load test samples from CSV. Returns list of dicts with 'text' key."""
    samples = []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["text"].strip():
                samples.append({"text": row["text"].strip(), "original_region": row.get("region", "")})
    print(f"Loaded {len(samples)} test samples")
    return samples


def generate_and_transcribe(tts_model, ref_audio, ref_text, gen_text, asr):
    """Generate audio with TTS, then transcribe with ASR. Returns ASR text."""
    wav, sr, _ = tts_model.infer(
        ref_file=ref_audio,
        ref_text=ref_text,
        gen_text=gen_text,
        nfe_step=32,
        cross_fade_duration=0.15,
        speed=1.0,
        seed=42,  # fixed seed for reproducibility
        remove_silence=False,
        show_info=lambda x: None,
    )
    # Save to temp WAV for ASR
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp_path = tmp.name
    sf.write(tmp_path, wav, sr)
    # Transcribe
    result = asr(tmp_path)
    os.unlink(tmp_path)
    return result["text"]


def load_tts_checkpoint(ckpt_step: int):
    """Load F5TTS with a specific checkpoint. Returns the model."""
    from f5_tts.api import F5TTS
    ckpt_path = os.path.join(CKPT_DIR, f"model_{ckpt_step}.safetensors")
    print(f"  Loading checkpoint: {ckpt_path}")
    tts = F5TTS(
        model="F5TTS_v1_Base",
        ckpt_file=ckpt_path,
        vocab_file=VOCAB_FILE,
        device="cuda",
    )
    return tts


def free_tts_memory(tts):
    """Delete TTS model and free GPU memory."""
    del tts
    gc.collect()
    torch.cuda.empty_cache()
    print("  GPU memory cleared.")


# Quick test
samples = load_test_samples(TESTS_CSV)
print(f"Sample 0: {samples[0]['text'][:80]}...")
print(f"Normalized: {normalize_text(samples[0]['text'])[:80]}...")

## 5. Main Evaluation Loop

In [ ]:
all_results = []  # list of dicts per sample
summary_results = []  # list of dicts per checkpoint x dialect

total_start = time.time()
total_gens = len(CHECKPOINTS) * len(DIALECTS) * len(samples)
gen_count = 0

for ckpt_step in CHECKPOINTS:
    print(f"\n{'='*60}")
    print(f"CHECKPOINT {ckpt_step}")
    print(f"{'='*60}")

    tts = load_tts_checkpoint(ckpt_step)

    for dialect in DIALECTS:
        print(f"\n  --- Dialect: {dialect} ---")
        cfg = DIALECT_CONFIG[dialect]
        ref_audio = cfg["ref_audio"]
        ref_text = cfg["ref_text"]

        dialect_refs = []
        dialect_hyps = []

        for i, sample in enumerate(samples):
            gen_count += 1
            original_text = sample["text"]

            try:
                asr_text = generate_and_transcribe(
                    tts, ref_audio, ref_text, original_text, asr_pipe
                )
            except Exception as e:
                print(f"    [ERROR] Sample {i}: {e}")
                asr_text = ""

            norm_ref = normalize_text(original_text)
            norm_hyp = normalize_text(asr_text)

            if norm_ref and norm_hyp:
                sample_wer = compute_wer(norm_ref, norm_hyp)
            else:
                sample_wer = 1.0  # treat empty as 100% error

            all_results.append({
                "checkpoint": ckpt_step,
                "dialect": dialect,
                "sample_idx": i,
                "original_region": sample["original_region"],
                "original_text": original_text,
                "asr_text": asr_text,
                "norm_original": norm_ref,
                "norm_asr": norm_hyp,
                "wer": sample_wer,
            })

            dialect_refs.append(norm_ref)
            dialect_hyps.append(norm_hyp)

            if (i + 1) % 10 == 0 or i == len(samples) - 1:
                elapsed = time.time() - total_start
                rate = gen_count / elapsed
                eta = (total_gens - gen_count) / rate if rate > 0 else 0
                print(
                    f"    [{gen_count}/{total_gens}] "
                    f"Sample {i+1}/{len(samples)} | "
                    f"WER: {sample_wer:.2%} | "
                    f"ETA: {eta/60:.1f} min"
                )

        # Compute dialect-level WER
        valid = [(r, h) for r, h in zip(dialect_refs, dialect_hyps) if r and h]
        if valid:
            refs_valid, hyps_valid = zip(*valid)
            dialect_wer = compute_wer(list(refs_valid), list(hyps_valid))
        else:
            dialect_wer = 1.0

        summary_results.append({
            "checkpoint": ckpt_step,
            "dialect": dialect,
            "num_samples": len(samples),
            "num_valid": len(valid),
            "wer": dialect_wer,
        })
        print(f"  >> {dialect} WER @ ckpt {ckpt_step}: {dialect_wer:.2%}")

    free_tts_memory(tts)

total_elapsed = time.time() - total_start
print(f"\n{'='*60}")
print(f"DONE! Total time: {total_elapsed/60:.1f} min ({total_elapsed/3600:.1f} hrs)")
print(f"Total generations: {gen_count}")

## 6. Export Results

In [ ]:
import pandas as pd

# Detailed results
df_detail = pd.DataFrame(all_results)
detail_path = os.path.join(OUTPUT_DIR, "wer_detailed_results.csv")
df_detail.to_csv(detail_path, index=False)
print(f"Detailed results saved: {detail_path}")

# Summary results
df_summary = pd.DataFrame(summary_results)
summary_path = os.path.join(OUTPUT_DIR, "wer_summary.csv")
df_summary.to_csv(summary_path, index=False)
print(f"Summary results saved: {summary_path}")

# Copy to Drive
import shutil
shutil.copy2(detail_path, DRIVE_RESULTS)
shutil.copy2(summary_path, DRIVE_RESULTS)
print(f"Results copied to Drive: {DRIVE_RESULTS}")

# Display summary table
pivot = df_summary.pivot(index="checkpoint", columns="dialect", values="wer")
pivot["Overall"] = df_detail.groupby("checkpoint")["wer"].mean()
print("\n" + "="*60)
print("WER SUMMARY (lower is better)")
print("="*60)
print(pivot.to_string(float_format="{:.2%}".format))

## 7. Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 12, "figure.dpi": 120})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Line plot: WER across checkpoints per dialect ---
ax1 = axes[0]
colors = {"North": "#4A90D9", "Central": "#E8833A", "South": "#50B86C"}
for dialect in DIALECTS:
    sub = df_summary[df_summary["dialect"] == dialect]
    ax1.plot(
        sub["checkpoint"], sub["wer"] * 100,
        marker="o", linewidth=2, markersize=8,
        label=dialect, color=colors[dialect],
    )
# Overall
overall = df_detail.groupby("checkpoint")["wer"].mean()
ax1.plot(
    overall.index, overall.values * 100,
    marker="s", linewidth=2.5, markersize=8,
    label="Overall", color="#333333", linestyle="--",
)
ax1.set_xlabel("Checkpoint (training steps)")
ax1.set_ylabel("Word Error Rate (%)")
ax1.set_title("WER Across Checkpoints")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xticks(CHECKPOINTS)
ax1.set_xticklabels([f"{c//1000}k" for c in CHECKPOINTS])

# --- Heatmap: Checkpoint x Dialect ---
ax2 = axes[1]
pivot_pct = pivot[DIALECTS] * 100
im = ax2.imshow(pivot_pct.values, cmap="RdYlGn_r", aspect="auto")
ax2.set_xticks(range(len(DIALECTS)))
ax2.set_xticklabels(DIALECTS)
ax2.set_yticks(range(len(CHECKPOINTS)))
ax2.set_yticklabels([f"{c//1000}k" for c in CHECKPOINTS])
ax2.set_xlabel("Dialect")
ax2.set_ylabel("Checkpoint")
ax2.set_title("WER Heatmap (%)")
# Annotate cells
for i in range(len(CHECKPOINTS)):
    for j in range(len(DIALECTS)):
        val = pivot_pct.values[i, j]
        ax2.text(j, i, f"{val:.1f}%", ha="center", va="center",
                 fontsize=11, fontweight="bold",
                 color="white" if val > 50 else "black")
plt.colorbar(im, ax=ax2, label="WER (%)")

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "wer_results.png")
plt.savefig(fig_path, bbox_inches="tight")
shutil.copy2(fig_path, DRIVE_RESULTS)
plt.show()
print(f"Figure saved to: {fig_path}")